# 스파이크 후속: 피처 엔지니어링을 추가하면 XGBoost-로지스틱 격차가 벌어지는가

배경: 튜닝 후에도 XGBoost와 로지스틱 회귀의 AUC 차이(+0.0094)와 상위 5% 포착 성능 차이(정밀도 +0.5%p)가 작았다.
`docs/project-plan.md` Phase 2에 적힌 파생 변수 후보(연체 총합, 최악 연체 단계, 소득 결측 여부, 1인당 소득)를
추가했을 때 두 모델의 격차가 벌어지는지 확인한다.

이론적으로 예상해볼 점: 트리 모델(XGBoost)은 원본 변수들의 비선형·상호작용을 스스로 찾아낼 수 있는 반면
선형 모델(로지스틱 회귀)은 그런 관계를 명시적인 파생 변수로 넣어줘야 잡아낸다. 따라서 파생 변수 추가는
**로지스틱 회귀 쪽을 더 끌어올려 격차를 좁힐 가능성**도 있다 — 실제로 어느 쪽인지 실험으로 확인한다.

범위: 여전히 스파이크다. 여기서 좋은 결과가 나온 파생 변수라도 Phase 2에서 데이터 누수·재현성을 다시 점검한 뒤 채택 여부를 정한다.


In [1]:
import time
from contextlib import contextmanager

import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import randint, uniform
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RANDOM_STATE = 42
DATA_DIR = "../../data/raw"
TARGET = "SeriousDlqin2yrs"
DELINQ_COLS = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]

timings = {}


@contextmanager
def timer(step_name):
    start = time.perf_counter()
    yield
    elapsed = time.perf_counter() - start
    timings[step_name] = elapsed
    print(f"[{step_name}] {elapsed:.1f}초")


## 1. 로드 + 최소 전처리 + 파생 변수

이전 스파이크와 동일한 최소 전처리(96/98 코드값 → 결측 → 중앙값 대체, winsorize)에 아래 파생 변수를 추가한다.

- `IncomeMissingFlag`: 원본 `MonthlyIncome` 결측 여부 (대체 전에 기록)
- `TotalPastDue`: 연체 변수 3개(코드값 정리 후) 합계
- `WorstDelinquencyStage`: 도달한 최악 연체 단계 (0=없음, 1=30~59일, 2=60~89일, 3=90일 이상)
- `IncomePerDependent`: `MonthlyIncome / (NumberOfDependents + 1)`


In [2]:
with timer("로드+전처리+피처엔지니어링"):
    train = pd.read_csv(f"{DATA_DIR}/cs-training.csv", index_col=0)
    df = train.copy()

    # 파생 변수는 원본 결측/코드값 정보가 남아있을 때 만들어야 하므로 대체 이전에 계산
    df["IncomeMissingFlag"] = df["MonthlyIncome"].isna().astype(int)

    for col in DELINQ_COLS:
        df.loc[df[col] >= 96, col] = np.nan

    df["TotalPastDue"] = df[DELINQ_COLS].sum(axis=1, skipna=True)
    df["WorstDelinquencyStage"] = np.select(
        [
            df["NumberOfTimes90DaysLate"].fillna(0) > 0,
            df["NumberOfTime60-89DaysPastDueNotWorse"].fillna(0) > 0,
            df["NumberOfTime30-59DaysPastDueNotWorse"].fillna(0) > 0,
        ],
        [3, 2, 1],
        default=0,
    )

    missing_cols = ["MonthlyIncome", "NumberOfDependents"] + DELINQ_COLS
    df[missing_cols] = df[missing_cols].fillna(df[missing_cols].median())

    df["IncomePerDependent"] = df["MonthlyIncome"] / (df["NumberOfDependents"] + 1)

    feature_cols = [c for c in df.columns if c != TARGET]
    lower = df[feature_cols].quantile(0.005)
    upper = df[feature_cols].quantile(0.995)
    df[feature_cols] = df[feature_cols].clip(lower=lower, upper=upper, axis=1)

print("피처 수:", len(feature_cols))
print("추가된 피처:", [c for c in feature_cols if c not in train.columns])
df[feature_cols].describe()


[로드+전처리+피처엔지니어링] 0.1초
피처 수: 14
추가된 피처: ['IncomeMissingFlag', 'TotalPastDue', 'WorstDelinquencyStage', 'IncomePerDependent']


,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,IncomeMissingFlag,TotalPastDue,WorstDelinquencyStage,IncomePerDependent
count,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000
mean,0.322328,52.285793,0.238687,325.217939,6200.663493,8.427433,0.081107,1.007767,0.059593,0.734747,0.198207,0.391313,0.340287,4448.016962
std,0.356769,14.720757,0.647630,955.301614,4134.774785,5.025920,0.382234,1.043331,0.278261,1.093439,0.398650,1.027402,0.781922,3243.162290
min,0.000000,23.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.029867,41.000000,0.000000,0.175074,3903.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2151.000000
50%,0.154181,52.000000,0.000000,0.366508,5400.000000,8.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4000.000000
75%,0.559046,63.000000,0.000000,0.868254,7400.000000,11.000000,0.000000,2.000000,0.000000,1.000000,0.000000,0.000000,0.000000,5400.000000
max,1.366269,89.000000,4.000000,6186.010000,31250.000000,27.000000,3.000000,6.000000,2.000000,5.000000,1.000000,7.000000,3.000000,22500.000000


## 2. 학습/홀드아웃 분할 (이전 스파이크와 동일한 random_state로 분할 재현)

In [3]:
X = df[feature_cols]
y = df[TARGET]

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print("학습:", X_train.shape, "홀드아웃:", X_holdout.shape)


학습: (120000, 14) 홀드아웃: (30000, 14)


## 3. 로지스틱 회귀 — 하이퍼파라미터 탐색 (피처 엔지니어링 포함)

In [4]:
logreg_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
logreg_grid = {
    "clf__C": [0.001, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100],
    "clf__class_weight": [None, "balanced"],
}

with timer("로지스틱 회귀 탐색"):
    logreg_search = GridSearchCV(logreg_pipe, logreg_grid, scoring="roc_auc", cv=cv, n_jobs=-1)
    logreg_search.fit(X_train, y_train)

logreg_best = logreg_search.best_estimator_
logreg_proba = logreg_best.predict_proba(X_holdout)[:, 1]
logreg_auc = roc_auc_score(y_holdout, logreg_proba)
print("최적 파라미터:", logreg_search.best_params_)
print("홀드아웃 AUC:", round(logreg_auc, 4))


[로지스틱 회귀 탐색] 3.0초
최적 파라미터: {'clf__C': 0.001, 'clf__class_weight': None}
홀드아웃 AUC: 0.8619


## 4. XGBoost — 하이퍼파라미터 탐색 (피처 엔지니어링 포함)

In [5]:
pos_weight_ratio = (y_train == 0).sum() / (y_train == 1).sum()
xgb_param_dist = {
    "n_estimators": randint(100, 400),
    "max_depth": randint(2, 8),
    "learning_rate": uniform(0.01, 0.29),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "min_child_weight": randint(1, 10),
    "scale_pos_weight": [1, 5, 10, round(pos_weight_ratio, 2), 20],
}

with timer("XGBoost 탐색"):
    xgb_search = RandomizedSearchCV(
        xgb.XGBClassifier(objective="binary:logistic", eval_metric="auc", random_state=RANDOM_STATE),
        param_distributions=xgb_param_dist,
        n_iter=40,
        scoring="roc_auc",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    xgb_search.fit(X_train, y_train)

xgb_best = xgb_search.best_estimator_
xgb_proba = xgb_best.predict_proba(X_holdout)[:, 1]
xgb_auc = roc_auc_score(y_holdout, xgb_proba)
print("최적 파라미터:", xgb_search.best_params_)
print("홀드아웃 AUC:", round(xgb_auc, 4))


[XGBoost 탐색] 21.7초
최적 파라미터: {'colsample_bytree': np.float64(0.6888431241882921), 'learning_rate': np.float64(0.044760956526768016), 'max_depth': 5, 'min_child_weight': 8, 'n_estimators': 259, 'scale_pos_weight': 1, 'subsample': np.float64(0.7615344684232164)}
홀드아웃 AUC: 0.8693


## 5. AUC 비교: 피처 엔지니어링 전 vs 후

In [6]:
# 이전 스파이크(spike_tuning_comparison.ipynb) 결과 — 원본 변수만 사용
PREV_LOGREG_AUC = 0.8598
PREV_XGB_AUC = 0.8691

comparison = pd.DataFrame(
    [
        ("로지스틱 회귀", PREV_LOGREG_AUC, logreg_auc, logreg_auc - PREV_LOGREG_AUC),
        ("XGBoost", PREV_XGB_AUC, xgb_auc, xgb_auc - PREV_XGB_AUC),
    ],
    columns=["모델", "AUC (원본 변수)", "AUC (피처엔지니어링 후)", "개선폭"],
)
comparison


,모델,AUC (원본 변수),AUC (피처엔지니어링 후),개선폭
0,로지스틱 회귀,0.8598,0.861926,0.002126
1,XGBoost,0.8691,0.869307,0.000207


In [7]:
gap_before = PREV_XGB_AUC - PREV_LOGREG_AUC
gap_after = xgb_auc - logreg_auc
print(f"피처 엔지니어링 전 격차(XGB-LR): {gap_before:+.4f}")
print(f"피처 엔지니어링 후 격차(XGB-LR): {gap_after:+.4f}")
print(f"격차 변화: {gap_after - gap_before:+.4f} ({'벌어짐' if gap_after > gap_before else '좁혀짐'})")


피처 엔지니어링 전 격차(XGB-LR): +0.0093
피처 엔지니어링 후 격차(XGB-LR): +0.0074
격차 변화: -0.0019 (좁혀짐)


## 6. 고위험 상위 5% 포착 성능 비교

In [8]:
def top_k_metrics(y_true, proba, k_ratio=0.05):
    n = len(y_true)
    k = int(np.ceil(n * k_ratio))
    order = np.argsort(-proba)
    top_idx = order[:k]
    y_true_arr = np.asarray(y_true)
    n_bad_in_top = y_true_arr[top_idx].sum()
    precision = n_bad_in_top / k
    recall = n_bad_in_top / y_true_arr.sum()
    lift = precision / y_true_arr.mean()
    return precision, recall, lift

logreg_p, logreg_r, logreg_l = top_k_metrics(y_holdout, logreg_proba)
xgb_p, xgb_r, xgb_l = top_k_metrics(y_holdout, xgb_proba)

# 이전 스파이크(원본 변수) 결과
PREV_LOGREG_P, PREV_LOGREG_R = 0.4747, 0.3551
PREV_XGB_P, PREV_XGB_R = 0.4800, 0.3591

top5_comparison = pd.DataFrame(
    [
        ("로지스틱 회귀", "정밀도", PREV_LOGREG_P, logreg_p),
        ("로지스틱 회귀", "포착률", PREV_LOGREG_R, logreg_r),
        ("XGBoost", "정밀도", PREV_XGB_P, xgb_p),
        ("XGBoost", "포착률", PREV_XGB_R, xgb_r),
    ],
    columns=["모델", "지표", "원본 변수", "피처엔지니어링 후"],
)
top5_comparison["변화"] = top5_comparison["피처엔지니어링 후"] - top5_comparison["원본 변수"]
top5_comparison


,모델,지표,원본 변수,피처엔지니어링 후,변화
0,로지스틱 회귀,정밀도,0.4747,0.473333,-0.001367
1,로지스틱 회귀,포착률,0.3551,0.354115,-0.000985
2,XGBoost,정밀도,0.4800,0.483333,0.003333
3,XGBoost,포착률,0.3591,0.361596,0.002496


## 7. XGBoost 피처 중요도 — 파생 변수가 실제로 쓰이는가

In [9]:
importance = pd.Series(xgb_best.feature_importances_, index=feature_cols).sort_values(ascending=False)
new_features = ["IncomeMissingFlag", "TotalPastDue", "WorstDelinquencyStage", "IncomePerDependent"]
print("파생 변수 중요도 순위:")
for feat in new_features:
    rank = list(importance.index).index(feat) + 1
    print(f"  {feat}: 중요도 {importance[feat]:.4f} (전체 {len(feature_cols)}개 중 {rank}위)")
print()
importance


파생 변수 중요도 순위:
  IncomeMissingFlag: 중요도 0.0081 (전체 14개 중 11위)
  TotalPastDue: 중요도 0.1390 (전체 14개 중 3위)
  WorstDelinquencyStage: 중요도 0.3668 (전체 14개 중 1위)
  IncomePerDependent: 중요도 0.0079 (전체 14개 중 13위)



WorstDelinquencyStage                   0.366785
NumberOfTimes90DaysLate                 0.223641
TotalPastDue                            0.139027
NumberOfTime30-59DaysPastDueNotWorse    0.068952
RevolvingUtilizationOfUnsecuredLines    0.060995
NumberOfTime60-89DaysPastDueNotWorse    0.056641
NumberOfOpenCreditLinesAndLoans         0.016130
NumberRealEstateLoansOrLines            0.014731
age                                     0.012733
DebtRatio                               0.009771
IncomeMissingFlag                       0.008077
MonthlyIncome                           0.007918
IncomePerDependent                      0.007912
NumberOfDependents                      0.006689
dtype: float32

In [10]:
timing_df = pd.DataFrame([(k, f"{v:.1f}초") for k, v in timings.items()], columns=["단계", "소요 시간"])
print(f"전체 합계: {sum(timings.values()):.1f}초")
timing_df


전체 합계: 24.9초


,단계,소요 시간
0,로드+전처리+피처엔지니어링,0.1초
1,로지스틱 회귀 탐색,3.0초
2,XGBoost 탐색,21.7초


## 8. 결론

- 5절의 "격차 변화"가 양수면 파생 변수가 XGBoost를 더 유리하게 만든 것이고, 음수면 로지스틱 회귀를 더 따라잡게 한 것이다.
- 7절에서 파생 변수의 중요도 순위가 낮으면(하위권) 이번에 고른 파생 변수들이 원본 변수가 이미 담고 있던 정보를
  재조합한 것에 가까워 새 정보량이 적었다는 뜻이다.
- 결과는 `docs/spike-feasibility.md`에 반영한다. 여기서 격차가 크게 벌어지지 않는다면, 이 데이터셋에서는
  XGBoost 우위가 피처 엔지니어링으로도 뚜렷해지지 않는다는 근거가 하나 더 쌓이는 것이다.
